# Lab 06 — Forecasting

用同時段 baseline 和 rolling residual 建立 prediction interval，進行 forecasting anomaly 和 capacity early warning。

In [ ]:
from pathlib import Path
from IPython.display import SVG, display

def find_project_root(start=Path.cwd()):
    start = start.resolve()
    for base in (start, *start.parents):
        if (base / "environment.yml").exists():
            return base
    raise FileNotFoundError("Could not find project root containing environment.yml")

svg_candidates = [
    Path("diagrams/lab06_pipeline_position.svg"),
    find_project_root() / "labs/self-study" / "diagrams/lab06_pipeline_position.svg",
]
for svg_path in svg_candidates:
    if svg_path.exists():
        display(SVG(filename=str(svg_path)))
        break
else:
    raise FileNotFoundError("Could not find diagram: diagrams/lab06_pipeline_position.svg")


## Lab 06 概覽

### 學習目標

1. 理解「預測型異常偵測」與「過去比較型異常偵測」的根本差異
2. 實作 rolling forecast baseline 與 prediction interval
3. 理解季節性對 forecast accuracy 的影響
4. 比較 rolling baseline and SARIMA / LSTM 方法的 tradeoff

### 前置條件

Lab 01 完成（`data/processed/features.csv` 存在）

## 設計主線：預測要服務反應時間

本章的實務連結是提前處理。Forecasting 的價值不在於把未來畫得很漂亮，而是在 SLA 受影響前給工程師足夠時間行動。

設計預測偵測時請問三個問題：

1. **horizon 是否可行**：預測 5 分鐘後、30 分鐘後或 2 小時後，對值班流程的意義不同。
2. **interval 是否對應風險**：區間太窄會誤報，太寬會漏報。高風險服務可以更敏感，低風險服務可以更保守。
3. **季節性是否穩定**：上週同時段是否真的能代表今天？節假日、搬遷、業務成長會破壞這個假設。

設計原則：forecasting 是營運提前量的設計，不只是時間序列模型選型。


In [ ]:
from pathlib import Path
import warnings
warnings.filterwarnings("ignore", category=FutureWarning)

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

plt.style.use("seaborn-v0_8-whitegrid")
pd.set_option("display.max_columns", 120)
pd.set_option("display.width", 160)

def show_fig(fig):
    display(fig)
    plt.close(fig)

PROJECT_ROOT = Path.cwd()
while PROJECT_ROOT != PROJECT_ROOT.parent:
    if (PROJECT_ROOT / "environment.yml").exists():
        break
    PROJECT_ROOT = PROJECT_ROOT.parent

DATA_SYNTHETIC = PROJECT_ROOT / "data" / "synthetic"
DATA_PROCESSED = PROJECT_ROOT / "data" / "processed"
REPORTS = PROJECT_ROOT / "reports"
DATA_PROCESSED.mkdir(parents=True, exist_ok=True)
REPORTS.mkdir(parents=True, exist_ok=True)

print(f"Project root: {PROJECT_ROOT}")

In [ ]:
features = pd.read_csv(DATA_PROCESSED / "features.csv", parse_dates=["timestamp"])
event_catalog = pd.read_csv(DATA_SYNTHETIC / "synthetic_event_catalog.csv", parse_dates=["start_time", "end_time"])

---
📖 **概念說明 ▸ 講師引導** — 講師帶領說明約 12 分鐘，請先不要執行 cell。

---

## 📖 概念：預測型偵測 vs 回顧型偵測

### 兩種異常偵測的根本區別

| | 回顧型（Labs 02–04） | 預測型（本 lab） |
|---|---|---|
| **問題** | 「目前的值和過去的自己相比，偏離了多少？」 | 「目前的值和預測值相比，偏離了多少？」 |
| **Baseline** | Rolling mean / median of past values | 模型預測的未來值 |
| **優點** | 簡單、快速、不需要建模 | 能考慮趨勢和季節性；更適合有規律的資料 |
| **缺點** | 無法區分「正常的週期性高峰」和「異常的高峰」| 需要建立準確的預測模型；預測誤差本身引入噪音 |

**具體例子**：週一早上 9 點的流量高峰

- 回顧型偵測（z-score）：「這個值比過去一小時的平均高 3σ」→ 觸發告警
- 預測型偵測：「預測週一早上 9 點流量高——模型知道這是正常的」→ 不觸發

預測型偵測能學到「哪些時段本來就應該是什麼水準」，減少由正常季節性引起的誤報。

### 季節性的三個尺度

網路流量的季節性通常有多個週期疊加：

| 週期 | 名稱 | 典型驅動因素 |
|---|---|---|
| 24 小時 | 日週期（Diurnal） | 上班時間 vs 夜間 vs 凌晨備份 |
| 7 天 | 週週期（Weekly） | 工作日 vs 週末 |
| 年 | 年週期（Annual） | 年底結帳、假日 |

本課程的 rolling forecast 使用**同一週、同一小時**的歷史平均作為預測——
這隱含地捕捉了日週期和週週期，而不需要顯式建模季節性。

### Rolling Forecast 的運作方式

```
預測 t 時刻的值：
    → 找出過去 N 週，與 t 相同的「星期幾 + 小時」的所有歷史值
    → 取這些值的 mean ± k × std 作為預測區間
    → 若實際值落在區間外，標記為異常
```

這是一種 **naïve seasonal decomposition**：簡單但對日 / 週週期效果好。
它在沒有趨勢（流量長期穩定）的情況下特別有效。

### 三種預測方法比較預告

| 方法 | 假設 | 優點 | 缺點 |
|---|---|---|---|
| Rolling historical average | 同時段的歷史重複 | 無需訓練、可解釋 | 對長期趨勢變化無感 |
| SARIMA | 線性趨勢 + 季節性 | 有嚴謹的統計基礎 | 需要平穩性（Stationarity）假設 |
| LSTM | 任意非線性模式 | 理論上能學到複雜模式 | 需要大量資料、不可解釋 |

## Step 1 - 建立 forecast baseline 與 prediction interval

---
💻 **自行執行** — 閱讀說明並依序執行以下 cell。有疑問請舉手。

---

In [ ]:
target = "traffic_total"
rows = []
for port_id, g in features.groupby("port_id", sort=False):
    g = g.sort_values("timestamp").copy()
    g["slot"] = g["timestamp"].dt.dayofweek.astype(str) + "-" + g["timestamp"].dt.hour.astype(str) + "-" + (g["timestamp"].dt.minute // 5).astype(str)
    seasonal = g.groupby("slot")[target].transform(lambda s: s.shift(1).rolling(4, min_periods=1).mean())
    moving = g[target].shift(1).rolling(12, min_periods=3).mean()
    g["y_hat"] = seasonal.fillna(moving).fillna(g[target].expanding().mean())
    residual = (g[target] - g["y_hat"]).abs()
    band = residual.shift(1).rolling(24, min_periods=6).quantile(0.95).fillna(residual.quantile(0.95))
    g["y_hat_lower"] = (g["y_hat"] - band).clip(lower=0)
    g["y_hat_upper"] = g["y_hat"] + band
    capacity = g[target].quantile(0.995) * 1.15
    g["capacity"] = capacity
    g["forecast_positive_anomaly"] = g[target] > g["y_hat_upper"]
    g["forecast_negative_anomaly"] = g[target] < g["y_hat_lower"]
    g["forecast_30m"] = g["y_hat"].shift(-6)
    g["early_warning_30m"] = g["forecast_30m"] > capacity * 0.80
    rows.append(g)

forecast = pd.concat(rows, ignore_index=True)
display(forecast[["timestamp", "port_id", target, "y_hat", "y_hat_lower", "y_hat_upper", "early_warning_30m"]].head())

---
💻 **自行執行** — 閱讀說明並依序執行以下 cell。有疑問請舉手。

---

## Step 2 - 視覺化：Actual vs Forecast + prediction interval

In [ ]:
port_id = "port-id7431"
p = forecast[forecast["port_id"] == port_id].copy()

fig, ax = plt.subplots(figsize=(14, 5))
ax.plot(p["timestamp"], p[target], label="actual", linewidth=1)
ax.plot(p["timestamp"], p["y_hat"], label="forecast", linewidth=1.2)
ax.fill_between(p["timestamp"], p["y_hat_lower"], p["y_hat_upper"], alpha=0.18, label="prediction interval")
ax.plot(p["timestamp"], p["capacity"], label="capacity", color="tab:red", linestyle="--")
warnings_df = p[p["early_warning_30m"]]
ax.scatter(warnings_df["timestamp"], warnings_df[target], color="tab:orange", s=18, label="early warning")
for _, event in event_catalog.iterrows():
    if event["port_id"] in [port_id, "MULTI"]:
        ax.axvspan(event["start_time"], event["end_time"], alpha=0.10, color="tab:red")
ax.set_title(f"{port_id} - Forecasting and capacity early warning")
ax.legend(loc="upper left")
ax.grid(alpha=0.25)
plt.tight_layout()
show_fig(fig)

---
💻 **自行執行** — 閱讀說明並依序執行以下 cell。有疑問請舉手。

---

## Step 3 - 視覺化：Forecasting anomaly summary

In [ ]:
summary = (
    forecast.groupby("event_label")
    .agg(
        rows=("timestamp", "size"),
        positive_anomaly=("forecast_positive_anomaly", "sum"),
        negative_anomaly=("forecast_negative_anomaly", "sum"),
        early_warning=("early_warning_30m", "sum"),
    )
)
summary["positive_rate"] = summary["positive_anomaly"] / summary["rows"]
display(summary.sort_values("positive_rate", ascending=False))

fig, ax = plt.subplots(figsize=(12, 5))
summary["early_warning"].sort_values().plot(kind="barh", ax=ax, color="tab:orange")
ax.set_title("Early warning count by event label")
ax.set_xlabel("count")
plt.tight_layout()
show_fig(fig)

In [ ]:
keep = [
    "timestamp", "device_id", "port_id", "port_role", "event_label", "event_id",
    target, "y_hat_lower", "y_hat", "y_hat_upper", "capacity",
    "forecast_positive_anomaly", "forecast_negative_anomaly", "forecast_30m", "early_warning_30m",
]
out_path = DATA_PROCESSED / "forecast_results.csv"
forecast[keep].to_csv(out_path, index=False)
print(f"saved: {out_path}")

---

## ⚖️ 方法比較：Rolling Baseline vs SARIMA vs LSTM

| | Rolling Historical Average | SARIMA | LSTM |
|---|---|---|---|
| **平穩性需求** | 不需要（隱含假設周期穩定）| 嚴格需要（差分後平穩）| 不需要（資料量夠大時）|
| **趨勢處理** | 差（無法捕捉長期增長）| 好（MA 部分處理趨勢）| 好 |
| **季節性** | 好（直接用同時段歷史）| 好（S in SARIMA 處理季節項）| 理論上好，但需要大量資料 |
| **訓練時間** | 幾乎不需要 | 分鐘級（per metric）| 小時級（per metric，需 GPU）|
| **可解釋性** | 高（「和上週同時段比」）| 中（統計係數可解釋）| 低 |
| **資料需求** | 1–4 週 | 幾個月（需要多個季節週期）| 數月到數年 |

### 何時應該升級到 SARIMA 或 LSTM？

升級 SARIMA 的條件：
- 網路流量有明顯的長期增長趨勢（每月增加 10%+）
- 有多個季節週期（日 + 週 + 年）且都很規律

升級 LSTM 的條件：
- 有 2 年以上的歷史資料
- 流量模式複雜（多個非線性模式疊加）
- 有 GPU 資源和 MLOps 基礎設施維護模型

**本課程的選擇**：Rolling baseline 在大多數企業網路環境中夠用。
它的主要弱點是「無法捕捉長期趨勢」——如果你的流量每個月穩定增長，
rolling baseline 的預測區間也會每個月放寬，最終失去偵測能力。

---
🔧 **探索練習** — 修改指定參數，觀察結果變化。

---

## 🔧 探索練習：Prediction Interval 寬度

在 Step 1 的 code cell 中找到控制 prediction interval 寬度的乘數（通常是 `k=2` 或 `multiplier=2`），
改成 `1.5`，重新執行 Step 2 的視覺化。

觀察：
- Prediction interval 變窄後，正常時段是否出現更多「超出區間」的點？
- 已知事件期間是否更明顯突出？
- 你認為 `k=2` 和 `k=1.5` 哪個在你的業務環境更適合？

---

> 「如果你的公司在過去 6 個月把所有業務從某個資料中心遷移到另一個，rolling forecast 的表現會有什麼問題？」

> 「Rolling baseline 用「上週同時段」作為預測。如果是一個全年無休但有長假（農曆年）的系統，這個假設會在哪裡失效？」

> 「預測誤差本身也是一個訊號——哪個 port 的預測誤差最大？這代表什麼？」

## ⚠ 人類驗證點 #6 — 預測型偵測的 tradeoff：準確度 vs 可解釋性

Forecasting 偵測異常的核心邏輯是：「目前的值比預測的高（或低）太多」。
但「預測」本身的準確度決定了整個系統的品質。

### 如何判斷

評估預測品質，看兩個指標：

1. **Prediction interval 寬度**：interval 過寬（±50%）代表模型不確定，會漏掉真實異常；過窄（±5%）代表太嚴格，正常波動也會觸發
2. **Forecast anomaly recall**：已知事件有多少落在「實際值超出 prediction interval」的時段？

### 讓你重新考慮的信號

- 正常時段每天觸發超過 5 個 forecast anomaly → prediction interval 太窄
- 已知的 `traffic_surge` 事件完全沒有觸發 forecast anomaly → 1 小時的 horizon 可能太短

### 🔧 探索練習

在 Step 1 的 code cell 找到以下參數：

```python
FORECAST_HORIZON = 12    # 12 個 5 分鐘點 = 1 小時。改成 24（2 小時），觀察 interval 如何變化
INTERVAL_MULTIPLIER = 2  # 改成 1.5（更嚴格）或 3.0（更寬鬆），觀察 anomaly rate 的變化
```

> 「Rolling forecast 和 z-score（Lab 02）偵測到的事件，有什麼不同？哪些事件只有 forecasting 抓到？」

> 「如果網路流量在最近 3 個月穩定增長（業務擴張），rolling mean 和 SARIMA 哪個更準？為什麼？」

> 「預測型偵測什麼時候會比回顧型更適合？」（引導：有強季節性、長期趨勢的情境）
